# IMOS realtime data to DwC Event Core - SMRU Example

Plan: Convert the realtime QCed IMOS marine mammal position data to DwC, and then publish the result to the IPT.

Contemporary notes from our meet w/ Ian Jonsen here: https://docs.google.com/document/d/1hibIxBbyGwa7b5-LRpnKIyjnr41EkUPKBzdUJoyAfaU/edit#heading=h.6bqw4binj5hq

### Inputs / configuration parameters:

* QCed data for a given campaign or project as the exported CSVs with appended position correction data as per https://github.com/ianjonsen/ArgosQC
* credentials for IPT and 
* corresponding project ID to associate new data with
* minimum quality hit to keep

In [2]:
import pandas as pd

metadata_df = pd.read_csv('input/imos_ct188/qc/aodn/metadata_ct188_nrt.csv')
loc_df = pd.read_csv('input/imos_ct188/qc/aodn/diag_ct188_nrt.csv')

In [3]:
metadata_df[0:10]

,sattag_program,device_id,ptt,body,device_wmo_ref,tag_type,common_name,species,release_longitude,release_latitude,...,release_date,recovery_date,age_class,sex,length,estimated_mass,actual_mass,state_country,qc_start_date,qc_end_date
0,ct188,ct188-Q1029-961-24,265887,15961,Q9902155,FTD_GEN_24L,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-25T00:00:00Z,NaN,juvenille,m,2.35,300,NaN,Australia,2025-06-11T17:00:00Z,2025-09-05T11:00:00Z
1,ct188,ct188-Q1029-971-24,265882,15971,Q9902156,FTD_GEN_24L,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-26T00:00:00Z,NaN,juvenille,m,2.37,350,NaN,Australia,2025-07-02T12:00:00Z,2025-10-16T12:00:00Z
2,ct188,ct188-Q1029-972-24,265880,15972,Q9902157,FTD_GEN_24L,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-26T00:00:00Z,NaN,juvenille,m,2.04,260,NaN,Australia,2025-06-24T12:00:00Z,2025-10-16T12:00:00Z
3,ct188,ct188-Q1029-974-24,265885,15974,Q9902158,FTD_GEN_24L,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-26T00:00:00Z,NaN,juvenille,m,1.85,190,NaN,Australia,2025-06-09T12:00:00Z,2025-08-21T12:00:00Z
4,ct188,ct188-Q1044-983-24,265898,15983,Q9902160,CTD_GEN_24A,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-27T00:00:00Z,NaN,juvenille,m,2.48,380,NaN,Australia,2025-05-31T16:00:00Z,2025-06-22T16:00:00Z
5,ct188,ct188-Q1044-985-24,265899,15985,Q9902161,CTD_GEN_24A,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-27T00:00:00Z,NaN,juvenille,m,2.20,250,NaN,Australia,2025-06-10T11:00:00Z,2025-10-16T11:00:00Z
6,ct188,ct188-Q1044-993-24,265904,15993,Q9902164,CTD_GEN_24A,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-27T00:00:00Z,NaN,juvenille,m,2.02,250,NaN,Australia,2025-07-02T12:00:00Z,2025-10-15T18:00:00Z
7,ct188,ct188-Q1029-960-24,265886,15960,Q9902154,FTD_GEN_24L,Southern elephant seal,Mirounga leonina,159.016667,-54.349999,...,2025-05-28T00:00:00Z,NaN,juvenille,m,2.06,220,NaN,Australia,2025-06-13T09:00:00Z,2025-06-29T21:00:00Z


## Metadata - create events and occurrences for each row

Process the metadata csv into Event Core (animal releases + tag attachments) + Occurrences (HumanObservations) + emofs for same (biological measurements are here)

In [4]:
metadata_df

# event entries: eventID = [body]-[release_date]
#                eventDate =  [release_date]
#                latitude = [release_latitude]
#                longitude = [release_longitude]
#                modified = current_date()
#                geodeticDatum = EPSG:4326
#                country = state_country  (error in current dataset, French Overseas Territory should be French Southern Lands)

column_map = {'release_date':'eventDate',
              'release_latitude':'decimalLatitude',
              'release_longitude':'decimalLongitude',
              'state_country':'country'}

event_df = metadata_df.rename(columns=column_map)
event_df['modified'] = pd.to_datetime('now', utc=True).round(freq='s')
# eventID is instrument serial number (body) + release datetime (eventDate)
event_df['eventID'] = event_df['body'].astype(str).str.cat(event_df['eventDate'].astype(str), sep='-')
event_df['geodeticDatum'] = 'EPSG:4326'
# Optional: truncate the extra columns from the core
event_df =  event_df[['eventID', 'eventDate', 'decimalLatitude', 'decimalLongitude', 'modified', 'geodeticDatum', 'country']]

In [5]:
event_df[0:5]

,eventID,eventDate,decimalLatitude,decimalLongitude,modified,geodeticDatum,country
0,15961-2025-05-25T00:00:00Z,2025-05-25T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia
1,15971-2025-05-26T00:00:00Z,2025-05-26T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia
2,15972-2025-05-26T00:00:00Z,2025-05-26T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia
3,15974-2025-05-26T00:00:00Z,2025-05-26T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia
4,15983-2025-05-27T00:00:00Z,2025-05-27T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia


In [6]:
# EMOFs to harvest
# for the release events
# instrument manufacturer and model  (SMRU + [tag type])
# PTT
# device id
# WMO ref


In [7]:
# occ ext. entries:    occurrenceID = [body]-[release_date]
#                      species = [species]
#                      sex = [sex]
#                      eventID = [body]-[release_date]
#                      organismID = [body]-[release_date]

occ_column_map = {'release_date':'eventDate',
                  'species':'scientificName'}
occ_df = metadata_df.rename(columns=occ_column_map)
occ_df['occurrenceID'] = occ_df['body'].astype(str).str.cat(occ_df['eventDate'].astype(str), sep='-')
occ_df['eventID'] = occ_df['body'].astype(str).str.cat(occ_df['eventDate'].astype(str), sep='-')
occ_df['organismID'] = occ_df['body'].astype(str).str.cat(occ_df['eventDate'].astype(str), sep='-')
occ_df['basisOfRecord'] = 'HumanObservation'
occ_df = occ_df[['occurrenceID', 'organismID','eventID', 'sex', 'scientificName', 'basisOfRecord']]

In [8]:
occ_df[0:5]

,occurrenceID,organismID,eventID,sex,scientificName,basisOfRecord
0,15961-2025-05-25T00:00:00Z,15961-2025-05-25T00:00:00Z,15961-2025-05-25T00:00:00Z,m,Mirounga leonina,HumanObservation
1,15971-2025-05-26T00:00:00Z,15971-2025-05-26T00:00:00Z,15971-2025-05-26T00:00:00Z,m,Mirounga leonina,HumanObservation
2,15972-2025-05-26T00:00:00Z,15972-2025-05-26T00:00:00Z,15972-2025-05-26T00:00:00Z,m,Mirounga leonina,HumanObservation
3,15974-2025-05-26T00:00:00Z,15974-2025-05-26T00:00:00Z,15974-2025-05-26T00:00:00Z,m,Mirounga leonina,HumanObservation
4,15983-2025-05-27T00:00:00Z,15983-2025-05-27T00:00:00Z,15983-2025-05-27T00:00:00Z,m,Mirounga leonina,HumanObservation


In [9]:
# EMOFs to harvest
# for the occurrences:
# sex
# length
# weight

In [10]:
# Create event and occurrence entries from the locations data file
# 
# Event entries:  eventID = organismID + date_detected
#                 latitude = ssm_lat if exists else lat
#                 longitude = ssm_lon if exists else lon
#                 eventDate = d_date
#                 geodeticDatum = EPSG:4326
#                 coordinateUncertaintyInMeters = max(ssm_x_se, ssm_y_se)  -- 1 SE or 2 SE?
#                 

# add the relevant columns to loc_df from metadata_df to create organismID
loc_df = loc_df.merge(metadata_df[['device_id', 'body', 'release_date', 'species']], 
                      how='left', left_on='ref', right_on='device_id')


In [11]:
# combine the organismID + the detection date into the eventID
loc_df['eventID'] = loc_df['body'].astype(str).str.cat(loc_df[['release_date', 'd_date']], sep='-')

In [12]:

# Check: is this correct to do in all cases?
# where there has been no correction made (corrected positions = NA, 
#       then use the raw position data
loc_df['decimalLatitude'] = loc_df['ssm_lat'].fillna(loc_df['lat'])
loc_df['decimalLongitude'] = loc_df['ssm_lon'].fillna(loc_df['lon'])
loc_df['eventDate'] = loc_df['d_date']
loc_df['modified'] = pd.to_datetime('now', utc=True).round(freq='s') 

# constant
loc_df['geodeticDatum'] = 'EPSG:4326'

# Ian's got his ssm_x and ssm_y in km, not in m
loc_df['coordinateUncertaintyInMeters'] = loc_df[['ssm_x_se', 'ssm_y_se']].max(axis=1) * 1000

# revisit uncertainty - make a radius based on the max, but uncertainty is an ellipse
# OBIS doesn't know about it but we can make a Polygon and include it somewhere to preserve the better
# knowledge that we have.

# Where there are multiple hits for a given time step (many satellites have opinions on position at once), 

loc_df = loc_df.sort_values(['ref', 'd_date', 'lq'], ascending=False)
# drop all but the best of the location qualities
loc_df = loc_df.drop_duplicates(subset=['ref','d_date'], keep='first', inplace=False)


In [13]:
loc_df['lq'].unique()

array([-2, -1,  0,  1,  2,  3, -9], dtype=int64)

In [14]:
# fallback - where coordinateUncertaintyInMeters is still null (un-QCed), 
# let's do something with the class of fix from Argos. set a lookup table as in the ATN example?

# Ian recommends - dan costa, accuracy of argos locations at sea pinnipeds
# in that article they compared GPS to Argos locations.
# https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0008677

# While this paper recommends different regimes per species due to differences in surfacing behaviour, 
# we don't have that kind of broad data in real-time land.

# So we take their recommendations for marine mammals here, and maybe we'd leave the door open to use a non-mammal error chart
# for non-airbreathers

# Other methodologies have thrown out the A and B quality hits altogther. 
# I'm not opposed to doing that but i'll confirm it with the SME beforehand
# because the QC algorithm is re-positioning the bad hits for us already.
# and if they didn't throw them out, we may not want to either.

# 68th percentile location error distances from Costa et al, in metres
# old LQ designations - CLS moved to a Kalman-filtered location set in ~2011
# error ellipses are now de riguer - semimajor and 
# semiminor ellipse axis/orientation to quantify uncertainty
# So we could harvest those first.

error_table = {3:490,
               2:1010,
               1:1200,
               0:4180,
               -1:6190,
               -2:10280,
               -9:10280} # TODO : What is the corresponding code to -9 LQ? 
                         # AniMotum thinks it's a class B
missing_errors = loc_df['coordinateUncertaintyInMeters'].isna()
loc_df.loc[missing_errors, 'coordinateUncertaintyInMeters'] = loc_df.loc[missing_errors, 'lq'].map(error_table)

In [15]:
loc_df['coordinateUncertaintyInMeters'].describe()

count    12769.000000
mean      2622.448251
std       1919.658534
min        151.380000
25%       1428.915000
50%       2105.532000
75%       3128.564000
max      13232.347000
Name: coordinateUncertaintyInMeters, dtype: float64

In [16]:
loc_df['modified']

14501   2025-10-17 18:12:29+00:00
14497   2025-10-17 18:12:29+00:00
14496   2025-10-17 18:12:29+00:00
14495   2025-10-17 18:12:29+00:00
14494   2025-10-17 18:12:29+00:00
                   ...           
6826    2025-10-17 18:12:29+00:00
6825    2025-10-17 18:12:29+00:00
6824    2025-10-17 18:12:29+00:00
6823    2025-10-17 18:12:29+00:00
6822    2025-10-17 18:12:29+00:00
Name: modified, Length: 12769, dtype: datetime64[ns, UTC]

In [17]:
# Select the columns and append to the event_df

event_df = pd.concat([event_df, loc_df[['eventID', 'eventDate', 'decimalLatitude', 'decimalLongitude', 'modified','geodeticDatum', 'coordinateUncertaintyInMeters']]])

In [18]:
event_df

,eventID,eventDate,decimalLatitude,decimalLongitude,modified,geodeticDatum,country,coordinateUncertaintyInMeters
0,15961-2025-05-25T00:00:00Z,2025-05-25T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia,NaN
1,15971-2025-05-26T00:00:00Z,2025-05-26T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia,NaN
2,15972-2025-05-26T00:00:00Z,2025-05-26T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia,NaN
3,15974-2025-05-26T00:00:00Z,2025-05-26T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia,NaN
4,15983-2025-05-27T00:00:00Z,2025-05-27T00:00:00Z,-54.349999,159.016667,2025-10-17 18:12:18+00:00,EPSG:4326,Australia,NaN
...,...,...,...,...,...,...,...,...
6826,15960.0-2025-05-28T00:00:00Z-2025-05-28T17:06:18Z,2025-05-28T17:06:18Z,-54.499670,158.930520,2025-10-17 18:12:29+00:00,EPSG:4326,NaN,490.0
6825,15960.0-2025-05-28T00:00:00Z-2025-05-28T12:14:02Z,2025-05-28T12:14:02Z,-54.500190,158.957770,2025-10-17 18:12:29+00:00,EPSG:4326,NaN,1010.0
6824,15960.0-2025-05-28T00:00:00Z-2025-05-28T12:12:59Z,2025-05-28T12:12:59Z,-54.500720,158.922180,2025-10-17 18:12:29+00:00,EPSG:4326,NaN,490.0
6823,15960.0-2025-05-28T00:00:00Z-2025-05-28T07:20:40Z,2025-05-28T07:20:40Z,-54.490220,158.936910,2025-10-17 18:12:29+00:00,EPSG:4326,NaN,1200.0


In [19]:
# Occurrence entries: occurrenceID = eventID
#                     eventID = eventID
#                     species = species
#                     organismID = body + release_date

loc_df['occurrenceID'] = loc_df['eventID']
loc_df['organismID'] =  loc_df['body'].astype(str).str.cat(loc_df['release_date'].astype(str), sep='-') 

In [20]:
# Decimate to first each hour per animal. Acoustics would also use per-receiver location, argos and sat won't need that.
dets_df = loc_df
dets_df['scientificName'] = dets_df['species']
dets_df['basisOfRecord'] = 'MachineObservation'
dets_df['Date'] = pd.to_datetime(dets_df['d_date']).dt.date
dets_df['hr'] = pd.to_datetime(dets_df['d_date']).dt.hour
dets_df['binsize'] = dets_df.groupby(['organismID', 'Date', 'hr']).size().reset_index(name='binsize')['binsize']
dets_df.drop_duplicates(subset=['organismID','Date', 'hr'], keep='first', inplace=True)
dets_df.drop('hr', axis=1, inplace=True)
dets_df

,ref,ptt,d_date,lq,lat,lon,alt_lat,alt_lon,n_mess,n_mess_120,...,eventDate,modified,geodeticDatum,coordinateUncertaintyInMeters,occurrenceID,organismID,scientificName,basisOfRecord,Date,binsize
14501,ct188-Q1044-993-24,265904,2025-10-16T22:06:47Z,-2,-60.95583,160.05521,-60.95583,160.05520,2,1,...,2025-10-16T22:06:47Z,2025-10-17 18:12:29+00:00,EPSG:4326,10280.0,15993.0-2025-05-27T00:00:00Z-2025-10-16T22:06:47Z,15993.0-2025-05-27T00:00:00Z,Mirounga leonina,MachineObservation,2025-10-16,NaN
14497,ct188-Q1044-993-24,265904,2025-10-16T19:12:15Z,-2,-60.96012,159.87172,-60.96012,159.87172,2,0,...,2025-10-16T19:12:15Z,2025-10-17 18:12:29+00:00,EPSG:4326,10280.0,15993.0-2025-05-27T00:00:00Z-2025-10-16T19:12:15Z,15993.0-2025-05-27T00:00:00Z,Mirounga leonina,MachineObservation,2025-10-16,NaN
14496,ct188-Q1044-993-24,265904,2025-10-16T18:57:02Z,-2,-60.99056,159.84640,-60.99056,159.84640,1,0,...,2025-10-16T18:57:02Z,2025-10-17 18:12:29+00:00,EPSG:4326,10280.0,15993.0-2025-05-27T00:00:00Z-2025-10-16T18:57:02Z,15993.0-2025-05-27T00:00:00Z,Mirounga leonina,MachineObservation,2025-10-16,NaN
14495,ct188-Q1044-993-24,265904,2025-10-16T17:53:38Z,-2,-60.98826,159.84416,-60.98826,159.84416,1,0,...,2025-10-16T17:53:38Z,2025-10-17 18:12:29+00:00,EPSG:4326,10280.0,15993.0-2025-05-27T00:00:00Z-2025-10-16T17:53:38Z,15993.0-2025-05-27T00:00:00Z,Mirounga leonina,MachineObservation,2025-10-16,NaN
14493,ct188-Q1044-993-24,265904,2025-10-16T16:11:03Z,-2,-60.96657,159.84537,-60.96657,159.84537,2,0,...,2025-10-16T16:11:03Z,2025-10-17 18:12:29+00:00,EPSG:4326,10280.0,15993.0-2025-05-27T00:00:00Z-2025-10-16T16:11:03Z,15993.0-2025-05-27T00:00:00Z,Mirounga leonina,MachineObservation,2025-10-16,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6831,ct188-Q1029-960-24,265886,2025-05-29T03:14:38Z,-2,-54.47659,158.94003,-54.47659,158.94003,2,0,...,2025-05-29T03:14:38Z,2025-10-17 18:12:29+00:00,EPSG:4326,10280.0,15960.0-2025-05-28T00:00:00Z-2025-05-29T03:14:38Z,15960.0-2025-05-28T00:00:00Z,Mirounga leonina,MachineObservation,2025-05-29,1.0
6830,ct188-Q1029-960-24,265886,2025-05-28T22:22:33Z,-2,-54.50199,158.93927,-54.50199,158.93927,1,0,...,2025-05-28T22:22:33Z,2025-10-17 18:12:29+00:00,EPSG:4326,10280.0,15960.0-2025-05-28T00:00:00Z-2025-05-28T22:22:33Z,15960.0-2025-05-28T00:00:00Z,Mirounga leonina,MachineObservation,2025-05-28,1.0
6826,ct188-Q1029-960-24,265886,2025-05-28T17:06:18Z,3,-54.49967,158.93052,-54.49967,158.93053,11,0,...,2025-05-28T17:06:18Z,2025-10-17 18:12:29+00:00,EPSG:4326,490.0,15960.0-2025-05-28T00:00:00Z-2025-05-28T17:06:18Z,15960.0-2025-05-28T00:00:00Z,Mirounga leonina,MachineObservation,2025-05-28,2.0
6825,ct188-Q1029-960-24,265886,2025-05-28T12:14:02Z,2,-54.50019,158.95777,-54.50019,158.95776,18,4,...,2025-05-28T12:14:02Z,2025-10-17 18:12:29+00:00,EPSG:4326,1010.0,15960.0-2025-05-28T00:00:00Z-2025-05-28T12:14:02Z,15960.0-2025-05-28T00:00:00Z,Mirounga leonina,MachineObservation,2025-05-28,1.0


In [21]:
dets_df['binsize'].describe()

count    6095.000000
mean        1.362920
std         0.599094
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         8.000000
Name: binsize, dtype: float64

In [22]:
dets_df['dataGeneralizations'] = dets_df['binsize'].apply(lambda x: 'subsampled by hour, first of {} record(s)'.format(x))

In [23]:
occ_df = pd.concat([occ_df, dets_df[['occurrenceID', 'eventID', 'scientificName', 'organismID', 'basisOfRecord']]])

In [36]:
# flesh out the occurrence taxonomic entries with kingdom, phylum, class, order, family
import pyworms
import math

lookup_dict = {}
for name in occ_df['scientificName'].unique():
    if type(name)==str:
        resp = pyworms.aphiaRecordsByMatchNames(name)
        if len(resp[0]) == 0:
            print('\nNo match for name "{}"'.format(name))
            continue
        elif len(resp[0]) > 1:
            print('\nMultiple matches for name "{}"'.format(name))
            pprint.pprint(resp[0], indent=4)
            continue
        else:
            worms = resp[0][0]
            lookup_dict[name]={'scientificName': name,
                               'scientificNameID': worms['lsid'],
                               'taxonRank': worms['rank'],
                               'kingdom': worms['kingdom'],
                               'phylum': worms['phylum'],
                               'class': worms['class'],
                               'order': worms['order'],
                               'family': worms['family']}
        
lookup_df = pd.DataFrame.from_dict(lookup_dict, orient='index')

attempting to match Mirounga leonina
attempting to match nan


In [ ]:
lookup_df

In [ ]:
occ_df = occ_df.join(lookup_df, how='left', on='scientificName', rsuffix='_worms')

In [26]:
occ_df

,occurrenceID,organismID,eventID,sex,scientificName,basisOfRecord,scientificName_worms,scientificNameID,taxonRank,kingdom,phylum,class,order,family
0,196997-2023-12-21T00:00:00Z,196997-2023-12-21T00:00:00Z,196997-2023-12-21T00:00:00Z,m,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
1,153719-2023-12-23T00:00:00Z,153719-2023-12-23T00:00:00Z,153719-2023-12-23T00:00:00Z,m,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
2,221816-2023-12-21T00:00:00Z,221816-2023-12-21T00:00:00Z,221816-2023-12-21T00:00:00Z,m,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
3,221821-2023-12-21T00:00:00Z,221821-2023-12-21T00:00:00Z,221821-2023-12-21T00:00:00Z,m,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
4,196991-2024-01-09T00:00:00Z,196991-2024-01-09T00:00:00Z,196991-2024-01-09T00:00:00Z,f,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11,196997-2023-12-21T00:00:00Z-2023-12-21T21:03:26Z,196997-2023-12-21T00:00:00Z,196997-2023-12-21T00:00:00Z-2023-12-21T21:03:26Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
10,196997-2023-12-21T00:00:00Z-2023-12-21T20:13:51Z,196997-2023-12-21T00:00:00Z,196997-2023-12-21T00:00:00Z-2023-12-21T20:13:51Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
7,196997-2023-12-21T00:00:00Z-2023-12-21T19:26:17Z,196997-2023-12-21T00:00:00Z,196997-2023-12-21T00:00:00Z-2023-12-21T19:26:17Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
5,196997-2023-12-21T00:00:00Z-2023-12-21T18:33:18Z,196997-2023-12-21T00:00:00Z,196997-2023-12-21T00:00:00Z-2023-12-21T18:33:18Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae


In [27]:
occ_df['organismID'].unique()

array(['196997-2023-12-21T00:00:00Z', '153719-2023-12-23T00:00:00Z',
       '221816-2023-12-21T00:00:00Z', '221821-2023-12-21T00:00:00Z',
       '196991-2024-01-09T00:00:00Z', '221802-2023-12-20T00:00:00Z',
       '204698-2024-02-11T00:00:00Z', '204693-2023-12-22T00:00:00Z',
       '204688-2023-12-23T00:00:00Z', '204699-2024-02-08T00:00:00Z',
       '204703-2024-02-09T00:00:00Z', '204702-2023-12-20T00:00:00Z',
       '204701-2023-12-22T00:00:00Z', '204717-2024-02-09T00:00:00Z',
       '204704-2024-01-26T00:00:00Z', '204739-2023-12-23T00:00:00Z',
       '204732-2024-02-09T00:00:00Z', '204738-2023-12-31T00:00:00Z',
       '221824-2024-02-11T00:00:00Z', '221820-2023-12-19T00:00:00Z',
       '221798-2023-12-20T00:00:00Z', '221799-2024-02-11T00:00:00Z',
       '221801-2024-01-26T00:00:00Z', '221814-2023-12-19T00:00:00Z',
       '221810-2023-12-21T00:00:00Z'], dtype=object)

In [28]:
# Any EMOFs to harvest from detection occurrences?
# 
# 

In [29]:
# Push them out to files and an archive:

occ_df.to_csv('output/occurrences.csv', date_format='%Y-%m-%dT%H:%M:%S')
event_df.to_csv('output/events.csv', date_format='%Y-%m-%dT%H:%M:%S')
# emof_df.to_csv('output/emof.csv', date_format='%Y-%m-%dT%H:%M:%S')

In [30]:
# Zip and ship to an IPT

# Either via a form fill-in, or via depositing the archive on the IPT's filesystem?

# TODO: Try the form-fill first - use the OTN IPT workflows from ipython-utilities
# import requests  # session with the forms themselves
# import selenium  # or pick-n-click

### Debugging cells:

In [31]:
# throwing out all but max lq will help us de-duplicate these same-time-same-tag hits?
loc_df['lq'].describe()

count    35098.000000
mean        -1.341415
std          1.056915
min         -9.000000
25%         -2.000000
50%         -2.000000
75%         -1.000000
max          3.000000
Name: lq, dtype: float64